In [1]:
import polars as pl
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

df = pl.read_parquet('/home/montes/proyectos/paralela/polar_vrs_pandas/taxi_filtrado.parquet')
print(f'Dataset cargado: {df.shape[0]:,} filas')

Dataset cargado: 10,906,858 filas


#  Preparación de datos para ML

In [2]:
# Características base
features = ['trip_distance', 'fare_amount', 'tip_amount', 'tolls_amount', 'passenger_count']
target = 'total_amount'

# Agregar características creadas en el notebook anterior si existen
extra_features = ['pickup_hour', 'trip_duration_sec', 'cost_per_mile']
for col in extra_features:
    if col in df.columns:
        features.append(col)

# Eliminar filas con valores nulos en las columnas de interés
df_ml = df.drop_nulls(subset=features + [target])

X = df_ml.select(features).to_numpy()
y = df_ml.select(target).to_numpy().ravel()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

Train: 8,725,486 | Test: 2,181,372


# Entrenamiento y evaluación de modelos

In [3]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=50, random_state=42)
}

results = []

for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start

    start = time.time()
    y_pred = model.predict(X_test)
    pred_time = time.time() - start

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    results.append({
        'Modelo': name,
        'Tiempo_Entrenamiento_s': round(train_time, 4),
        'Tiempo_Prediccion_s': round(pred_time, 4),
        'MAE': round(mae, 4),
        'RMSE': round(rmse, 4)
    })

df_results = pl.DataFrame(results)
print(df_results)

shape: (3, 5)
┌───────────────────┬────────────────────────┬─────────────────────┬────────┬────────┐
│ Modelo            ┆ Tiempo_Entrenamiento_s ┆ Tiempo_Prediccion_s ┆ MAE    ┆ RMSE   │
│ ---               ┆ ---                    ┆ ---                 ┆ ---    ┆ ---    │
│ str               ┆ f64                    ┆ f64                 ┆ f64    ┆ f64    │
╞═══════════════════╪════════════════════════╪═════════════════════╪════════╪════════╡
│ Linear Regression ┆ 2.5529                 ┆ 0.1554              ┆ 0.3282 ┆ 0.3741 │
│ Random Forest     ┆ 580.1477               ┆ 21.113              ┆ 0.1812 ┆ 0.5268 │
│ Gradient Boosting ┆ 422.3625               ┆ 1.0794              ┆ 0.6289 ┆ 1.3377 │
└───────────────────┴────────────────────────┴─────────────────────┴────────┴────────┘


##
1. Mejor precisión predictiva: Random Forest
MAE más bajo (0.1812): Sus predicciones se desvían en promedio solo $0.18 del valor real
Pero tiene el mayor RMSE (0.5268), lo que indica que tiene algunos errores grandes (outliers)

2. Mejor velocidad: Linear Regression
Entrenamiento 260x más rápido que Random Forest (2.4s vs 621s)
Predicción 244x más rápida (0.08s vs 20.38s)
Precisión aceptable (MAE: 0.3282)

3. Peor desempeño: Gradient Boosting
MAE más alto (0.6289): Peor precisión
RMSE más alto (1.3377): Errores grandes frecuentes
Tiempo de entrenamiento alto (421s)